# 07a — Caracterización Nivel 1 (6 arquetipos identitarios)


In [1]:
import pandas as pd, numpy as np, joblib
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import euclidean_distances

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_QC = ROOT / 'data' / 'data_qc_gustos'
DM = ROOT / 'data' / 'data_models_gustos'
INFORMES = ROOT / 'informes' / 'fase4_gustos' / '05_archetypes' / 'nivel1'
(INFORMES / 'radar_charts').mkdir(parents=True, exist_ok=True)

assignments = pd.read_parquet(DATA_QC / 'two_stage_assignments.parquet')
features_tier1 = pd.read_parquet(DATA_QC / 'features_tier1.parquet')
df = features_tier1.merge(assignments[['user_id','nivel1_cluster','has_tier2']], on='user_id')
print(f'Sample: {df.shape}')
print(f'Cluster distribution:')
for k, n in df['nivel1_cluster'].value_counts().sort_index().items():
    print(f'  C{k}: {n:,} ({n/len(df)*100:.2f}%)')


Sample: (114412, 81)
Cluster distribution:
  C0: 7,492 (6.55%)
  C1: 20,931 (18.29%)
  C2: 5,989 (5.23%)
  C3: 5,291 (4.62%)
  C4: 67,308 (58.83%)
  C5: 7,401 (6.47%)


In [2]:
def characterize(cluster_id):
    cluster_df = df[df['nivel1_cluster']==cluster_id]
    rest_df = df[df['nivel1_cluster']!=cluster_id]
    rows = []
    for feat in df.columns:
        if feat in ['user_id','nivel1_cluster','has_tier2']: continue
        if not pd.api.types.is_numeric_dtype(df[feat]): continue
        c_med = cluster_df[feat].median()
        r_med = rest_df[feat].median()
        g_iqr = df[feat].quantile(0.75) - df[feat].quantile(0.25)
        dist = (c_med - r_med) / g_iqr if g_iqr > 0 else 0
        rows.append({'feature':feat, 'cluster_median':c_med, 'cluster_p25':cluster_df[feat].quantile(0.25),
                     'cluster_p75':cluster_df[feat].quantile(0.75), 'rest_median':r_med,
                     'global_median':df[feat].median(), 'distinctiveness':dist,
                     'abs_distinctiveness':abs(dist)})
    return pd.DataFrame(rows).sort_values('abs_distinctiveness', ascending=False)

characterizations = {}
all_stats = []
for k in sorted(df['nivel1_cluster'].unique()):
    st = characterize(k)
    st['cluster'] = k
    characterizations[k] = st
    all_stats.append(st)
    print(f'\nC{k} top 8 distintivas:')
    print(st.head(8)[['feature','cluster_median','rest_median','distinctiveness']].to_string(index=False))

# Tabla consolidada de estadísticas
pd.concat(all_stats, ignore_index=True).to_csv(INFORMES / 'archetype_statistics.csv', index=False)
print('\narchetype_statistics.csv guardado')



C0 top 8 distintivas:
                     feature  cluster_median  rest_median  distinctiveness
       coll_items_recent_90d         38.0000     0.000000         1.900000
    n_distinct_concepts_used          9.0000     5.000000         1.000000
         coll_pct_recent_60d          1.0000     0.000000         1.000000
pct_days_active_currency_30d          0.1000     0.033333         1.000000
               coll_age_days         17.0000   110.000000        -0.978947
    reward_register_age_days         17.0000   113.000000        -0.950495
          binge_index_fights          1.8125     1.000000         0.937500
       user_account_age_days         17.0000   116.000000        -0.925234



C1 top 8 distintivas:
                    feature  cluster_median  rest_median  distinctiveness
   n_distinct_concepts_used        5.000000     8.000000        -0.750000
         binge_index_fights        1.000000     1.565217        -0.652174
       binge_index_currency        1.000000     1.569426        -0.618941
      gini_currency_concept        0.423077     0.620085        -0.591213
          items_mean_attack      170.777778   121.615385         0.544682
        items_mean_critical        0.108696     0.026316         0.509489
   pct_items_critical_build        0.100000     0.027778         0.505556
char_equip_slots_filled_max        5.000000     4.000000         0.500000



C2 top 8 distintivas:
                 feature  cluster_median  rest_median  distinctiveness
  pct_items_high_enhance        0.222222     0.000000         6.000000
     items_mean_critical        0.931818     0.023810         5.615697
       items_mean_attack      629.137931   122.631579         5.611702
pct_items_critical_build        0.666667     0.025641         4.487179
items_mean_enhance_level        3.666667     1.142857         4.346137
char_critical_chance_max        8.000000     0.000000         4.000000
         char_attack_max     5389.040000   626.000000         3.888450
     char_experience_max     5478.000000   535.000000         3.369461



C3 top 8 distintivas:
                 feature  cluster_median  rest_median  distinctiveness
   user_account_age_days      431.000000   106.000000         3.037383
reward_register_age_days      330.000000   103.000000         2.247525
           coll_age_days      201.000000   101.000000         1.052632
items_mean_enhance_level        1.700000     1.142857         0.959430
entropy_currency_concept        1.028992     1.770951        -0.944367
  items_redundancy_ratio        1.428571     1.125000         0.910714
   gini_currency_concept        0.256944     0.534054        -0.831596
n_distinct_concepts_used        3.000000     6.000000        -0.750000



C4 top 8 distintivas:
                 feature  cluster_median  rest_median  distinctiveness
entropy_currency_concept             0.0     1.795357        -2.285136
   entropy_currency_type             0.0     1.095795        -2.187902
   gini_currency_concept             0.0     0.546296        -1.639413
n_distinct_concepts_used             1.0     6.000000        -1.250000
char_critical_chance_max             0.0     2.000000        -1.000000
    pct_chars_high_level             0.0     0.333333        -1.000000
pct_items_critical_build             0.0     0.129935        -0.909545
     items_mean_critical             0.0     0.142857        -0.883518



C5 top 8 distintivas:
                 feature  cluster_median  rest_median  distinctiveness
   user_account_age_days      444.000000   104.000000         3.177570
reward_register_age_days      339.000000   102.000000         2.346535
  pct_items_high_enhance        0.071429     0.000000         1.928571
items_mean_enhance_level        2.070000     1.142857         1.596590
 items_max_enhance_level        8.000000     2.000000         1.500000
    pct_chars_high_level        0.333333     0.000000         1.000000
char_critical_chance_max        2.000000     0.000000         1.000000
  items_redundancy_ratio        1.448276     1.125000         0.969828

archetype_statistics.csv guardado


In [3]:
# Radar charts
RADAR_FEATURES = [
    'char_count', 'char_level_max', 'coll_total_items', 'reward_claim_rate',
    'items_total_enhance_invested', 'char_has_arena', 'user_account_age_days', 'iap_is_payer'
]
RADAR_FEATURES = [f for f in RADAR_FEATURES if f in df.columns]
print(f'Radar features: {RADAR_FEATURES}')

# Normalizar usando P90 global
norms = {f: df[f].quantile(0.90) for f in RADAR_FEATURES}

def cluster_radar_vals(cluster_id):
    cdf = df[df['nivel1_cluster']==cluster_id]
    vals = []
    for f in RADAR_FEATURES:
        med = cdf[f].median()
        n = norms[f]
        vals.append(min(med/n, 1.5) if n>0 else 0)
    return vals

def plot_radar(values, labels, title, savepath, color='steelblue'):
    angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
    values = list(values) + [values[0]]
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True))
    ax.plot(angles, values, color=color, linewidth=2)
    ax.fill(angles, values, color=color, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, size=8)
    ax.set_ylim(0, 1.5)
    ax.set_title(title, size=12, weight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(savepath, dpi=120, bbox_inches='tight')
    plt.close()

colors = plt.cm.tab10(np.linspace(0, 1, 6))
all_values = {}
for k, color in zip(sorted(df['nivel1_cluster'].unique()), colors):
    n_k = (df['nivel1_cluster']==k).sum()
    vals = cluster_radar_vals(k)
    all_values[k] = vals
    plot_radar(vals, RADAR_FEATURES, f'Arquetipo {k} (N={n_k:,})',
               INFORMES / 'radar_charts' / f'archetype_{k}_radar.png', color)

# Overlay
fig, ax = plt.subplots(figsize=(10,10), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2*np.pi, len(RADAR_FEATURES), endpoint=False).tolist() + [0]
for k, color in zip(sorted(df['nivel1_cluster'].unique()), colors):
    n_k = (df['nivel1_cluster']==k).sum()
    vals = list(all_values[k]) + [all_values[k][0]]
    ax.plot(angles, vals, color=color, linewidth=2, label=f'C{k} (N={n_k:,})')
    ax.fill(angles, vals, color=color, alpha=0.1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(RADAR_FEATURES, size=9)
ax.set_ylim(0, 1.5)
ax.set_title('Comparación de los 6 arquetipos (Nivel 1)', size=14, weight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=9)
plt.tight_layout()
plt.savefig(INFORMES / 'radar_charts' / 'all_archetypes_overlay.png', dpi=120, bbox_inches='tight')
plt.close()
print('6 radars individuales + overlay guardados')


Radar features: ['char_count', 'char_level_max', 'coll_total_items', 'reward_claim_rate', 'items_total_enhance_invested', 'char_has_arena', 'user_account_age_days', 'iap_is_payer']


6 radars individuales + overlay guardados


In [4]:
# Top 5 features discriminantes (ratio: std(medianas)/std(global))
disc = {}
for feat in df.select_dtypes(include='number').columns:
    if feat in ['nivel1_cluster','has_tier2']: continue
    medians = df.groupby('nivel1_cluster')[feat].median()
    g_std = df[feat].std()
    if g_std > 0:
        disc[feat] = float(medians.std() / g_std)

top5 = sorted(disc.items(), key=lambda x: -x[1])[:5]
top5_feats = [f for f,_ in top5]
print(f'Top 5 discriminantes: {top5_feats}')

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, feat in zip(axes, top5_feats):
    data = [df[df['nivel1_cluster']==k][feat].dropna() for k in sorted(df['nivel1_cluster'].unique())]
    ax.boxplot(data, showfliers=False, labels=[f'C{k}' for k in sorted(df['nivel1_cluster'].unique())])
    ax.set_title(feat, fontsize=10)
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(INFORMES / 'boxplots_top_features.png', dpi=120, bbox_inches='tight')
plt.close()
print('boxplots_top_features.png')


Top 5 discriminantes: ['is_arena_player', 'reward_register_age_days', 'coll_items_recent_30d', 'coll_pct_recent_30d', 'pct_items_critical_build']


/var/folders/0f/3zq36dr1451_ttvt93n947ph0000gw/T/ipykernel_7218/3532150833.py:17: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, showfliers=False, labels=[f'C{k}' for k in sorted(df['nivel1_cluster'].unique())])
/var/folders/0f/3zq36dr1451_ttvt93n947ph0000gw/T/ipykernel_7218/3532150833.py:17: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, showfliers=False, labels=[f'C{k}' for k in sorted(df['nivel1_cluster'].unique())])
/var/folders/0f/3zq36dr1451_ttvt93n947ph0000gw/T/ipykernel_7218/3532150833.py:17: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, showfliers=False, labels

boxplots_top_features.png


In [5]:
# Ejemplos representativos (5 jugadores cercanos al centroide)
model = joblib.load(DM / 'nivel1_kmeans_k6.joblib')
preproc = joblib.load(DM / 'preprocessor_tier1.joblib')

# Preprocesar features tier1 (mismo orden que en entrenamiento)
X_scaled = preproc.transform(features_tier1.drop(columns=['user_id']))
centroids = model['model'].cluster_centers_

examples = []
for k in sorted(df['nivel1_cluster'].unique()):
    mask = (df['nivel1_cluster']==k).values
    cluster_X = X_scaled[mask]
    cluster_uids = df.loc[mask, 'user_id'].values
    distances = euclidean_distances(cluster_X, [centroids[k]]).flatten()
    top5 = np.argsort(distances)[:5]
    for i in top5:
        examples.append({'cluster': k, 'user_id': cluster_uids[i], 'distance_to_centroid': float(distances[i])})

pd.DataFrame(examples).to_csv(INFORMES / 'archetype_examples.csv', index=False)
print(f'archetype_examples.csv ({len(examples)} ejemplos)')


archetype_examples.csv (30 ejemplos)


In [6]:
from datetime import datetime

md = ['# Arquetipos identitarios (Nivel 1) — Perfiles narrativos', '',
      f'**Fecha**: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
      f'**Algoritmo**: KMeans K=6 sobre features Tier 1 (master_table_gustos_v3_aggressive)',
      f'**Sample**: {len(df):,} jugadores',
      f'**Silhouette**: 0.353',
      '', '## Resumen', '',
      '| C | N | % | % con Tier 2 |',
      '|---:|---:|---:|---:|']
for k in sorted(df['nivel1_cluster'].unique()):
    cdf = df[df['nivel1_cluster']==k]
    md.append(f"| {k} | {len(cdf):,} | {len(cdf)/len(df)*100:.2f}% | {cdf['has_tier2'].mean()*100:.1f}% |")

md += ['', '---', '']
for k in sorted(df['nivel1_cluster'].unique()):
    cdf = df[df['nivel1_cluster']==k]
    st = characterizations[k]
    md += [f'## Arquetipo {k}', '',
           f'- **N**: {len(cdf):,} ({len(cdf)/len(df)*100:.2f}% del sample)',
           f'- **Cobertura Tier 2** (actividad reciente): {cdf["has_tier2"].mean()*100:.1f}%',
           '', '### Top 12 features distintivas', '',
           '| Feature | Mediana cluster | Mediana resto | Distinctiveness |',
           '|---|---:|---:|---:|']
    for _, row in st.head(12).iterrows():
        md.append(f"| `{row['feature']}` | {row['cluster_median']:.2f} | {row['rest_median']:.2f} | {row['distinctiveness']:+.2f} |")
    md += ['', '### Interpretación tentativa', '',
           '_(rellenar manualmente con narrativa, ver `archetype_naming.md`)_', '', '---', '']

(INFORMES / 'archetype_profiles.md').write_text('\n'.join(md))
print('archetype_profiles.md guardado')


archetype_profiles.md guardado


In [7]:
# distinctive_features.md
md = ['# Features distintivas por arquetipo (Nivel 1)', '']
for k in sorted(df['nivel1_cluster'].unique()):
    st = characterizations[k]
    md.append(f'## Arquetipo {k}\n')
    md.append('### Sobre-representado (cluster > resto):\n')
    md.append('| Feature | C_med | R_med | Δ |\n|---|---:|---:|---:|')
    over = st[st['distinctiveness']>0].head(8)
    for _, r in over.iterrows():
        md.append(f"| `{r['feature']}` | {r['cluster_median']:.2f} | {r['rest_median']:.2f} | {r['distinctiveness']:+.2f} |")
    md.append('\n### Sub-representado (cluster < resto):\n')
    md.append('| Feature | C_med | R_med | Δ |\n|---|---:|---:|---:|')
    under = st[st['distinctiveness']<0].head(8)
    for _, r in under.iterrows():
        md.append(f"| `{r['feature']}` | {r['cluster_median']:.2f} | {r['rest_median']:.2f} | {r['distinctiveness']:+.2f} |")
    md.append('\n---\n')

(INFORMES / 'distinctive_features.md').write_text('\n'.join(md))
print('distinctive_features.md')


distinctive_features.md
